In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

In [4]:
if not Path("input.txt").exists():
    raise FileNotFoundError("input.txt 파일을 Colab 왼쪽 파일 탭에 업로드하세요.")

text = open("input.txt", "r", encoding="utf-8").read()

print("전체 글자 수:", len(text))
print("앞부분 미리보기:")
print(text[:500])

전체 글자 수: 6213
앞부분 미리보기:
주가는 기업의 실적, 금리, 환율, 투자 심리, 수급 상황 등 여러 요인의 영향을 받는다.
코스피는 한국 유가증권시장의 대표적인 주가지수로 활용된다.
코스닥은 기술주와 성장주가 많이 포함된 한국의 대표적인 주식시장 지수다.
주식시장에서 투자자는 기업의 미래 이익 가능성을 고려하여 매수와 매도를 결정한다.
금리가 상승하면 기업의 자금 조달 비용이 증가할 수 있다.
금리가 하락하면 주식과 같은 위험자산에 대한 투자 매력이 높아질 수 있다.
환율은 수출기업과 수입기업의 실적 전망에 영향을 줄 수 있다.
원달러 환율이 상승하면 원화 가치가 달러 대비 하락했다는 의미다.
원달러 환율이 하락하면 원화 가치가 달러 대비 상승했다는 의미다.
외국인 투자자의 매수와 매도 흐름은 국내 증시에 영향을 줄 수 있다.
기관 투자자의 수급은 대형주 주가 흐름에 영향을 줄 수 있다.
개인 투자자의 거래 비중이 높아지면 단기 변동성이 커질 수 있다.
반도체 업종은 한국 주식시장에서 큰 비중을 차지하는 대표 업종


In [5]:
chars = sorted(list(set(text)))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("사용된 고유 글자 수 vocab_size:", vocab_size)
print("전체 데이터 길이:", len(data))
print("사용된 글자 일부:", chars[:50])

사용된 고유 글자 수 vocab_size: 381
전체 데이터 길이: 6213
사용된 글자 일부: ['\n', ' ', ',', '.', '0', '2', '5', 'B', 'E', 'F', 'G', 'O', 'P', 'R', 'S', 'T', '가', '각', '간', '감', '값', '강', '같', '개', '거', '건', '검', '것', '게', '격', '결', '경', '계', '고', '골', '곱', '공', '과', '관', '광', '교', '구', '국', '권', '규', '균', '그', '글', '금', '급']


In [6]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


block_size = 64
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))

print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)
print("입력 예시:", "".join(itos[i.item()] for i in xb[0][:40]))
print("정답 예시:", "".join(itos[i.item()] for i in yb[0][:40]))

xb.shape: torch.Size([64, 64])
yb.shape: torch.Size([64, 64])
입력 예시: 가한 금액이다.
평가손익은 현재 평가금액과 매입금액의 차이를 의미한다.

정답 예시: 한 금액이다.
평가손익은 현재 평가금액과 매입금액의 차이를 의미한다.
수


In [7]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()

        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)

        # 미래 글자는 보지 못하게 가림
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

In [8]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()

        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)

        # 미래 글자는 보지 못하게 가림
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()

        head_size = emb_dim // num_heads

        self.heads = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [10]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)

        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        # residual connection
        x = x + self.sa(self.ln1(x))

        # residual connection
        x = x + self.ffwd(self.ln2(x))

        return x

In [11]:
class TinyGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        emb_dim=128,
        num_heads=4,
        num_layers=4,
        dropout=0.1
    ):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)

        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape

        pos = torch.arange(T, device=x.device)

        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None, :, :]

        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)

        logits = self.lm_head(h)

        return logits


model = TinyGPT(vocab_size, block_size)
logits = model(xb)

print("logits.shape:", logits.shape)
print("정답 yb.shape:", yb.shape)

logits.shape: torch.Size([64, 64, 381])
정답 yb.shape: torch.Size([64, 64])


In [12]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)


def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()

    total_loss = 0.0
    total_count = 0

    for step, (xb, yb) in enumerate(loader):
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)

        if max_steps is not None and step + 1 >= max_steps:
            break

    return total_loss / total_count

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

torch.manual_seed(42)

model = TinyGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    emb_dim=128,
    num_heads=4,
    num_layers=4,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

loss_history = []

for epoch in range(30):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=100)
    loss_history.append(train_loss)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

device: cpu
epoch  0 | train loss 4.1231
epoch  1 | train loss 2.9811
epoch  2 | train loss 2.4772
epoch  3 | train loss 2.1040
epoch  4 | train loss 1.7317
epoch  5 | train loss 1.3827
epoch  6 | train loss 1.0810
epoch  7 | train loss 0.8326
epoch  8 | train loss 0.6398
epoch  9 | train loss 0.4994
epoch 10 | train loss 0.4033
epoch 11 | train loss 0.3364
epoch 12 | train loss 0.2909
epoch 13 | train loss 0.2557
epoch 14 | train loss 0.2301
epoch 15 | train loss 0.2106
epoch 16 | train loss 0.1959
epoch 17 | train loss 0.1833
epoch 18 | train loss 0.1732
epoch 19 | train loss 0.1652
epoch 20 | train loss 0.1579
epoch 21 | train loss 0.1518
epoch 22 | train loss 0.1460
epoch 23 | train loss 0.1410
epoch 24 | train loss 0.1373
epoch 25 | train loss 0.1331
epoch 26 | train loss 0.1303
epoch 27 | train loss 0.1286
epoch 28 | train loss 0.1242
epoch 29 | train loss 0.1232


In [14]:
@torch.no_grad()
def sample_gpt(
    model,
    block_size,
    stoi,
    itos,
    device,
    start_text="코스피는",
    max_new_tokens=300,
    temperature=0.9
):
    model.eval()

    context = torch.zeros((1, block_size), dtype=torch.long, device=device)

    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], dtype=torch.long, device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)

    out = list(start_text)

    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=-1)

        ix = torch.multinomial(probs, num_samples=1)

        out.append(itos[ix.item()])

        context = torch.cat([context[:, 1:], ix], dim=1)

    return "".join(out)

In [15]:
samples = []

start_texts = ["코스피는", "삼성전자는", "환율은", "투자자는"]

for start in start_texts:
    result = sample_gpt(
        model,
        block_size,
        stoi,
        itos,
        device,
        start_text=start,
        max_new_tokens=300,
        temperature=0.8
    )

    samples.append((start, result))

    print("=" * 80)
    print(f"start_text = {start}")
    print(result)

start_text = 코스피는
코스피는 한국 유가증권시장의 대표적인 주가지수로 활용된다.
코스닥은 기술주와 성장주가 많이 포함된 한국의 대표적인 주식시장 지수다.
주식시장에는 기업의 미래 이익 가능성을 고려하여 매수와 매도를 결정한다.
금리가 상승하면 기업의 자금 조달 비용이 증가할 수 있다.
금리가 하락하면 주식과 같은 위험자산에 대한 투자 매력이 높아질 수 있다.
환율은 수출기업과 수입기업의 실적 전망에 영향을 줄 수 있다.
원달러 환율이 상승하면 원화 가치가 달러 대비 하락했다는 의미다.
원달러 환율이 하락하면 원화 가치가 달러 대비 상승했다는 의미다.
외국인 투
start_text = 삼성전자는
삼성전자는 기업의 성장성뿐 아니라 재무 안정성도 함께 확인해야 한다.
단기 주가 흐름은 뉴스와 수급에 크게 흔들릴 수 있다.
장기 주가 흐름은 기업의 이익 성장과 밀접한 관련이 있다.
기술적 분석은 가격과 거래량의 패턴을 이용해 시장을 해석하는 방법이다.
기본적 분석은 기업의 실적과 재무상태를 바탕으로 가치를 평가하는 방법이다.
이동평균선은 일정 기간의 평균 가격을 연결한 지표다.
5일 이동평균선은 비교적 짧은 기간의 가격 흐름을 보여준다.
20일 이동평균선은 한 달 내외의 가격 흐름을 파악하는 데 자주 활용된다.
단기 이동평균선이 장기 이
start_text = 환율은
환율은 수출기업과 수입기업의 실적 전망에 영향을 줄 수 있다.
원달러 환율이 상승하면 원화 가치가 달러 대비 하락했다는 의미다.
원달러 환율이 하락하면 원화 가치가 달러 대비 상승했다는 의미다.
외국인 투자자의 매수와 매도 흐름은 국내 증시에 영향을 줄 수 있다.
기관 투자자의 수급은 대형주 주가 흐름에 영향을 줄 수 있다.
개인 투자자의 거래 비중이 높아지면 단기 변동성이 커질 수 있다.
반도체 업종은 한국 주식시장에서 큰 비중을 차지하는 대표 업종이다.
자동차 업종은 수출, 환율, 글로벌 수요의 영향을 받는다.
은행주는 금리 변화와 
start_text = 투자자는
투자자는 시장 전체의 흐

In [16]:
with open("generated_samples.txt", "w", encoding="utf-8") as f:
    for start, result in samples:
        f.write("=" * 80 + "\n")
        f.write(f"start_text = {start}\n\n")
        f.write(result + "\n\n")

print("generated_samples.txt 저장 완료")

generated_samples.txt 저장 완료


## 결과 분석

본 실험에서는 수업시간에 사용한 TinyGPT 구조를 유지하되, Tiny Shakespeare 데이터셋 대신 한국어 금융 뉴스 스타일 문장으로 구성한 `input.txt`를 학습 데이터로 사용했다.

학습 과정에서 train loss가 감소하는 것을 확인할 수 있었고, 이는 모델이 앞 글자를 바탕으로 다음 글자를 예측하는 능력을 점차 학습하고 있음을 의미한다.

생성 결과에서는 `코스피`, `삼성전자`, `환율`, `투자자`, `금리`, `주가`와 같은 금융 관련 표현이 나타났다. 이는 모델이 학습 데이터에 반복적으로 등장한 표현과 문체를 어느 정도 반영했음을 보여준다.

다만 데이터셋의 크기가 작고 문자 단위 모델이기 때문에 생성 문장에는 반복, 비문, 사실과 맞지 않는 표현이 포함될 수 있다. 따라서 본 모델은 실제 금융 조언을 제공하는 모델이 아니라, GPT 구조와 데이터셋 구성의 관계를 이해하기 위한 수업용 실험 모델이다.